<a href="https://colab.research.google.com/github/ksuplee/tensorflow-nlp-tutorial/blob/main/13_AI_Agent/13_03_LLM_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13_03 LLM 에이전트 : ReAct 루프와 함수 호출(Function Calling)

**학습 목표**
- 에이전트의 **사고(Reasoning) → 행동(Action) → 관찰(Observation)** 루프를 직접 구현해 본다.
- 에이전트가 **도구(tool)** 를 호출해 여러 단계를 자율적으로 수행하는 과정을 관찰한다.
- LLM API의 **함수 호출(Function Calling)** 개념을 이해한다.

> ※ 본 실습은 **개념·데모 수준**입니다. 외부 API 키 없이도 1~2단계가 실행되며, 심화 구현은 **'AI 에이전트 개발'** 교과목에서 다룹니다.

## 1. 도구(Tools) 정의

에이전트가 호출할 수 있는 함수들을 정의합니다. 각 함수는 하나의 **도구**입니다. 앞서 배운 **RAG(검색)도 하나의 도구**로 볼 수 있습니다(`search`).

In [1]:
import re

def calculator(expression: str) -> str:
    """사칙연산 문자열을 계산한다. 예: '25*4' -> '100'"""
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "오류: 허용되지 않은 문자"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"오류: {e}"

_WEATHER_DB = {"서울": "맑음, 28도", "부산": "흐림, 26도", "제주": "비, 24도"}
def get_weather(city: str) -> str:
    """도시의 현재 날씨(모의 데이터)를 반환한다."""
    return _WEATHER_DB.get(city, "정보 없음")

_KB = {
    "트랜스포머": "트랜스포머는 2017년 발표된 셀프 어텐션 기반 모델이다.",
    "bert": "BERT는 양방향 인코더로 사전학습된 언어모델이다.",
}
def search(query: str) -> str:
    """간단한 지식베이스에서 관련 문장을 찾는다(RAG의 검색도 도구의 하나)."""
    for k, v in _KB.items():
        if k in query.lower():
            return v
    return "검색 결과 없음"

# 도구 레지스트리: 이름 -> 함수
TOOLS = {"calculator": calculator, "get_weather": get_weather, "search": search}
print("등록된 도구:", list(TOOLS.keys()))

등록된 도구: ['calculator', 'get_weather', 'search']


## 2. ReAct 루프 직접 구현 (API 없이)

실제 에이전트에서는 **LLM이 다음 행동을 '추론'**합니다. 여기서는 원리를 눈으로 확인하기 위해 LLM 자리에 **간단한 규칙 기반 플래너**(`plan_next_action`)를 넣어 루프의 동작을 모사합니다.

In [2]:
CITIES = ["서울", "부산", "제주", "인천", "대구"]
OP = {"곱하기": "*", "더하기": "+", "빼기": "-", "나누기": "/"}

def plan_next_action(goal, history):
    """LLM을 대신하는 규칙 기반 플래너. 다음에 호출할 (도구, 인자, 사고)를 반환한다."""
    used = {h["tool"] for h in history}
    # 1) 날씨 요청
    if "날씨" in goal and "get_weather" not in used:
        city = next((c for c in CITIES if c in goal), "서울")
        return ("get_weather", city, f"'{city}'의 날씨를 확인해야 한다.")
    # 2) 사칙연산 요청
    m = re.search(r"(\d+)\s*(곱하기|더하기|빼기|나누기)\s*(\d+)", goal)
    if m and "calculator" not in used:
        expr = f"{m.group(1)}{OP[m.group(2)]}{m.group(3)}"
        return ("calculator", expr, f"'{expr}' 를 계산해야 한다.")
    # 3) 더 할 일이 없으면 종료
    return (None, None, "필요한 정보를 모두 모았으니 최종 답변을 정리한다.")

def react_agent(goal, max_steps=5):
    """사고 -> 행동 -> 관찰 루프를 반복하며 목표를 수행한다."""
    history = []
    print(f"[목표] {goal}\n")
    for step in range(1, max_steps + 1):
        tool, arg, thought = plan_next_action(goal, history)
        print(f"[{step}] 사고(Thought): {thought}")
        if tool is None:
            break
        obs = TOOLS[tool](arg)
        history.append({"tool": tool, "arg": arg, "obs": obs})
        print(f"    행동(Action): {tool}({arg!r})")
        print(f"    관찰(Observation): {obs}\n")
    answer = " / ".join(f"{h['arg']} → {h['obs']}" for h in history)
    print(f"[최종 답변] {answer}")
    return answer

**실행:** 검색·계산 두 단계가 필요한 질문을 주면, 에이전트가 스스로 순서대로 도구를 호출합니다.

In [3]:
_ = react_agent("25 곱하기 4는 얼마이고, 서울 날씨는 어때?")

[목표] 25 곱하기 4는 얼마이고, 서울 날씨는 어때?

[1] 사고(Thought): '서울'의 날씨를 확인해야 한다.
    행동(Action): get_weather('서울')
    관찰(Observation): 맑음, 28도

[2] 사고(Thought): '25*4' 를 계산해야 한다.
    행동(Action): calculator('25*4')
    관찰(Observation): 100

[3] 사고(Thought): 필요한 정보를 모두 모았으니 최종 답변을 정리한다.
[최종 답변] 서울 → 맑음, 28도 / 25*4 → 100


> 💡 **관찰 포인트**: 한 번의 답이 아니라 **여러 단계(사고→행동→관찰)를 반복**하며 필요한 도구를 골라 호출하는 것이 에이전트의 핵심입니다.

## 3. Hugging Face 모델을 이용한 도구 추천 (API 없이)

여기서는 LLM의 함수 호출(Function Calling) 기능 대신, Hugging Face 모델을 사용하여 사용자 질문의 '의도'를 파악하고, 그 의도에 따라 에이전트가 어떤 도구를 호출할지 '추천'하는 방법을 알아봅니다. 모델은 함수를 직접 실행하지 않고, 호출을 **결정**만 합니다. 실제 실행은 개발자 코드가 담당합니다.

> 아래 셀은 `transformers` 라이브러리를 사용하여 Hugging Face 모델을 로드하고 사용합니다.

In [5]:
# 먼저 설치: !pip install -q transformers
from transformers import pipeline
import json

# Hugging Face zero-shot classification pipeline 로드
# 모델은 문장의 의도를 분류하여 적절한 도구를 추천하는 데 사용됩니다.
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 에이전트가 사용할 수 있는 도구 목록을 후보 레이블로 정의합니다.
# 이 레이블들은 에이전트의 'tool' 정의와 일치해야 합니다.
candidate_labels = ["날씨 조회", "계산", "검색"] # 'get_weather', 'calculator', 'search' 에 대응

user_query = "오늘 서울 날씨는 어때?"

print(f"사용자 질문: '{user_query}'\n")

# Hugging Face 모델을 사용하여 질문의 의도를 분류합니다.
# 모델은 질문과 가장 일치하는 후보 레이블(도구)을 찾아냅니다.
classification_result = classifier(user_query, candidate_labels)

# 가장 높은 점수를 얻은 레이블(도구)을 선택합니다.
predicted_tool_intent = classification_result['labels'][0]
predicted_score = classification_result['scores'][0]

print(f"모델이 추천한 도구 의도: '{predicted_tool_intent}' (확률: {predicted_score:.2f})\n")

# 추천된 도구 의도에 따라 가상의 함수 호출을 시뮬레이션합니다.
# 실제 에이전트에서는 이 정보를 기반으로 TOOLS 딕셔너리의 함수를 호출합니다.
if predicted_tool_intent == "날씨 조회":
    # 질문에서 도시 이름을 추출하는 간단한 로직 (실제로는 더 정교한 NER 필요)
    city = "서울" if "서울" in user_query else "알 수 없음"
    tool_name = "get_weather"
    arguments = {"city": city}
    print(f"→ 에이전트가 '{tool_name}' 함수를 '{arguments}' 인자로 호출할 수 있습니다.")
    # 예시로 실제 함수를 호출하고 결과를 출력
    if city != "알 수 없음":
        print(f"    관찰(Observation): {TOOLS[tool_name](city)}")
elif predicted_tool_intent == "계산":
    # 질문에서 계산식을 추출하는 로직 (ReAct 루프의 plan_next_action 참조)
    expression = "25*4" # 예시 값
    tool_name = "calculator"
    arguments = {"expression": expression}
    print(f"→ 에이전트가 '{tool_name}' 함수를 '{arguments}' 인자로 호출할 수 있습니다.")
    print(f"    관찰(Observation): {TOOLS[tool_name](expression)}")
elif predicted_tool_intent == "검색":
    query = "트랜스포머" # 예시 값
    tool_name = "search"
    arguments = {"query": query}
    print(f"→ 에이전트가 '{tool_name}' 함수를 '{arguments}' 인자로 호출할 수 있습니다.")
    print(f"    관찰(Observation): {TOOLS[tool_name](query)}")
else:
    print("→ 적절한 도구를 찾지 못했습니다.")

print("\n---")
print("💡 **관찰 포인트**: Hugging Face 모델은 사용자 질문의 '의도'를 파악하여 어떤 도구를 사용할지 '추천'할 수 있습니다.")
print("    이는 LLM의 '함수 호출(Function Calling)'이 직접 JSON 형태의 호출 결정을 제공하는 것과는 다소 다른 방식입니다.")
print("    여기서는 모델의 추천을 바탕으로 개발자 코드가 실제 도구 호출을 담당하게 됩니다.")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

사용자 질문: '오늘 서울 날씨는 어때?'

모델이 추천한 도구 의도: '날씨 조회' (확률: 0.54)

→ 에이전트가 'get_weather' 함수를 '{'city': '서울'}' 인자로 호출할 수 있습니다.
    관찰(Observation): 맑음, 28도

---
💡 **관찰 포인트**: Hugging Face 모델은 사용자 질문의 '의도'를 파악하여 어떤 도구를 사용할지 '추천'할 수 있습니다.
    이는 LLM의 '함수 호출(Function Calling)'이 직접 JSON 형태의 호출 결정을 제공하는 것과는 다소 다른 방식입니다.
    여기서는 모델의 추천을 바탕으로 개발자 코드가 실제 도구 호출을 담당하게 됩니다.


## 4. 정리

| 개념 | 요약 |
|------|------|
| ReAct | 사고→행동→관찰을 반복하는 에이전트 루프 |
| Tool(도구) | 검색(RAG)·계산·코드 실행·외부 API 등 |
| Hugging Face 모델을 이용한 도구 추천 | 사용자 질문의 의도를 분류하여 에이전트가 호출할 도구를 '추천', 실행은 시스템이 담당 |

> 🔗 **연계**: 멀티 에이전트·플래닝·메모리·MCP·프로덕션 배포 등 심화는 **'AI 에이전트 개발'** 교과목에서 다룹니다.